# Benchmark: Baseline vs Optimized Inference

Compares the **baseline** revision (`07a3478f`) with the **optimized** branch (`claude/optimize-inference-speed-lkCRI`).

Tests **prune-aware + retain_kv_cache** mode with both **Flash Attention 2** and **SDPA** backends.

Metrics: output token match, wall time per generation, tokens/sec.

## 1. Setup

In [1]:
# ── Configuration ──────────────────────────────────────────────────────
REPO_URL = "https://github.com/anujjamwal/cognitive-compression.git"  # ← update if private/different
BASELINE_REV = "07a3478f3873ec6d8f2abb38c5073abe15033181"
OPTIMIZED_BRANCH = "claude/optimize-inference-speed-lkCRI"
MODEL_ID = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware"
MAX_NEW_TOKENS = 4096
NUM_SAMPLES = 5  # number of prompts to benchmark
BATCH_SIZE = 1   # keep 1 for clean wall-time comparison

In [2]:
!pip install transformers accelerate datasets trl math-verify --no-build-isolation -q

In [3]:
!nvcc --version
!python --version
import torch
print(f"Torch Version: {torch.__version__}")
print(f"CUDA Version: {torch.version.cuda}")


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Python 3.12.12
Torch Version: 2.10.0+cu128
CUDA Version: 12.8


In [4]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
!pip install --no-dependencies --upgrade flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

  Using cached https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl (253.6 MB)
Processing ./flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/content/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'



In [5]:
import os, sys, time, shutil, subprocess, gc, json
from pathlib import Path

import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"torch {torch.__version__}  |  CUDA {torch.version.cuda}  |  GPU: {torch.cuda.get_device_name(0)}")

torch 2.10.0+cu128  |  CUDA 12.8  |  GPU: NVIDIA A100-SXM4-40GB


In [6]:
import sys, os

# When running in Colab, clone the repo and add lib/ to the path
if "google.colab" in sys.modules:
    if not os.path.exists("cs224n-final-project"):
        !git clone https://github.com/anujjamwal/cs224n-final-project.git
    !cd cs224n-final-project && git checkout claude/optimize-inference-speed-lkCRI  &&  git pull --recurse-submodules
    sys.path.insert(0, "cs224n-final-project/lib")
else:
    # Local: notebook lives inside lib/ already
    sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

from custom_generate import generate

Branch 'claude/optimize-inference-speed-lkCRI' set up to track remote branch 'claude/optimize-inference-speed-lkCRI' from 'origin'.
Switched to a new branch 'claude/optimize-inference-speed-lkCRI'
Already up to date.


## 3. Load model & tokenizer (shared)

In [13]:
def load_model(attn_impl: str):
    """Load model with specified attention implementation."""
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, dtype=dtype, device_map="auto",
        attn_implementation=attn_impl,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    return model, tokenizer, dtype

## 4. Build test prompts

In [8]:
MATH_PROMPTS = [
    "Solve: What is the sum of all integer values of $n$ such that $\\frac{20}{2n-1}$ is an integer?",
    "If $f(x) = 3x^2 - 7$ and $g(x) = x + 1$, what is $f(g(2))$?",
    "Find the value of $x$ if $2^{x+1} = 32$.",
    "What is the remainder when $3^{100}$ is divided by 7?",
    "Simplify $\\sqrt{50} + \\sqrt{18}$.",
    "How many positive divisors does 360 have?",
    "If $\\log_2(x) + \\log_2(x-2) = 3$, find $x$.",
    "What is the coefficient of $x^3$ in the expansion of $(2x + 1)^5$?",
]

PROMPTS = MATH_PROMPTS[:NUM_SAMPLES]
print(f"Using {len(PROMPTS)} prompts for benchmark")

Using 5 prompts for benchmark


In [9]:
def tokenize_prompt(tokenizer, prompt: str) -> dict:
    """Build chat-formatted input for a single prompt."""
    messages = [
        {"role": "system", "content": "You are a helpful math assistant. Show your reasoning step by step. write your answer in \\boxed{}"},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt", padding=False)
    return {k: v.to("cuda") for k, v in enc.items()}

## 5. Benchmark helpers

In [15]:
def run_benchmark(model, tokenizer, generate_fn, prompts, label: str):
    """Run generation on all prompts and collect results."""
    results = []
    
    for i, prompt in enumerate(prompts):
        inp = tokenize_prompt(tokenizer, prompt)
        input_len = inp["input_ids"].shape[1]
        
        # Warmup on first prompt
        if i == 0:
            with torch.inference_mode():
                _ = model.generate(
                    **inp,
                    max_new_tokens=16,
                    processing_class=tokenizer,
                )
            torch.cuda.synchronize()
        
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        with torch.inference_mode():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_NEW_TOKENS,
                processing_class=tokenizer,
            )
        
        torch.cuda.synchronize()
        wall = time.perf_counter() - t0
        
        sequences = out if isinstance(out, torch.Tensor) else out.sequences
        gen_tokens = sequences[0, input_len:]
        num_gen = gen_tokens.shape[0]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=False)
        
        results.append({
            "prompt_idx": i,
            "label": label,
            "input_len": input_len,
            "gen_tokens": num_gen,
            "wall_sec": wall,
            "tok_per_sec": num_gen / wall if wall > 0 else 0,
            "token_ids": gen_tokens.cpu().tolist(),
            "decoded": decoded,
        })
        print(f"  [{label}] prompt {i}: {num_gen} tokens in {wall:.2f}s ({num_gen/wall:.1f} tok/s)")
    
    return results


def free_model():
    """Release GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

## 6. Run: Baseline (revision 07a3478f)

In [17]:
print("=" * 60)
print("BASELINE — SDPA")
print("=" * 60)

model_sdpa, tok_sdpa, dtype = load_model("sdpa")
gen_fn_baseline = None

baseline_sdpa_results = run_benchmark(model_sdpa, tok_sdpa, gen_fn_baseline, PROMPTS, "baseline-sdpa")

del model_sdpa, tok_sdpa
free_model()

BASELINE — SDPA


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [baseline-sdpa] prompt 0: 19 tokens in 39.97s (0.5 tok/s)
  [baseline-sdpa] prompt 1: 21 tokens in 3.77s (5.6 tok/s)
  [baseline-sdpa] prompt 2: 19 tokens in 7.69s (2.5 tok/s)
  [baseline-sdpa] prompt 3: 406 tokens in 81.36s (5.0 tok/s)
  [baseline-sdpa] prompt 4: 29 tokens in 38.09s (0.8 tok/s)


In [18]:
print("=" * 60)
print("BASELINE — Flash Attention 2")
print("=" * 60)

try:
    model_fa2, tok_fa2, dtype = load_model("flash_attention_2")
    gen_fn_baseline = None
    
    baseline_fa2_results = run_benchmark(model_fa2, tok_fa2, gen_fn_baseline, PROMPTS, "baseline-fa2")
    
    del model_fa2, tok_fa2
    free_model()
except Exception as e:
    print(f"FA2 not available: {e}")
    baseline_fa2_results = None

BASELINE — Flash Attention 2


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [baseline-fa2] prompt 0: 19 tokens in 46.83s (0.4 tok/s)
  [baseline-fa2] prompt 1: 21 tokens in 4.82s (4.4 tok/s)
  [baseline-fa2] prompt 2: 19 tokens in 9.77s (1.9 tok/s)
  [baseline-fa2] prompt 3: 437 tokens in 61.90s (7.1 tok/s)
  [baseline-fa2] prompt 4: 370 tokens in 39.08s (9.5 tok/s)


## 7. Run: Optimized (current branch)

In [19]:
def run_benchmark(model, tokenizer, generate_fn, prompts, label: str):
    """Run generation on all prompts and collect results."""
    results = []
    
    for i, prompt in enumerate(prompts):
        inp = tokenize_prompt(tokenizer, prompt)
        input_len = inp["input_ids"].shape[1]
        
        # Warmup on first prompt
        if i == 0:
            with torch.inference_mode():
                _ = model.generate(
                    **inp,
                    max_new_tokens=16,
                    custom_generate=generate_fn,
                    processing_class=tokenizer,
                )
            torch.cuda.synchronize()
        
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        
        with torch.inference_mode():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_NEW_TOKENS,
                custom_generate=generate_fn,
                processing_class=tokenizer,
            )
        
        torch.cuda.synchronize()
        wall = time.perf_counter() - t0
        
        sequences = out if isinstance(out, torch.Tensor) else out.sequences
        gen_tokens = sequences[0, input_len:]
        num_gen = gen_tokens.shape[0]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=False)
        
        results.append({
            "prompt_idx": i,
            "label": label,
            "input_len": input_len,
            "gen_tokens": num_gen,
            "wall_sec": wall,
            "tok_per_sec": num_gen / wall if wall > 0 else 0,
            "token_ids": gen_tokens.cpu().tolist(),
            "decoded": decoded,
        })
        print(f"  [{label}] prompt {i}: {num_gen} tokens in {wall:.2f}s ({num_gen/wall:.1f} tok/s)")
    
    return results

In [20]:
print("=" * 60)
print("OPTIMIZED — SDPA")
print("=" * 60)

model_sdpa, tok_sdpa, dtype = load_model("sdpa")
gen_fn_optimized = generate._sample

optimized_sdpa_results = run_benchmark(model_sdpa, tok_sdpa, gen_fn_optimized, PROMPTS, "optimized-sdpa")

del model_sdpa, tok_sdpa
free_model()

OPTIMIZED — SDPA


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [optimized-sdpa] prompt 0: 19 tokens in 39.69s (0.5 tok/s)
  [optimized-sdpa] prompt 1: 21 tokens in 3.75s (5.6 tok/s)
  [optimized-sdpa] prompt 2: 19 tokens in 7.63s (2.5 tok/s)
  [optimized-sdpa] prompt 3: 406 tokens in 81.09s (5.0 tok/s)
  [optimized-sdpa] prompt 4: 29 tokens in 37.87s (0.8 tok/s)


In [21]:
print("=" * 60)
print("OPTIMIZED — Flash Attention 2")
print("=" * 60)

try:
    model_fa2, tok_fa2, dtype = load_model("flash_attention_2")
    gen_fn_optimized = generate._sample
    
    optimized_fa2_results = run_benchmark(model_fa2, tok_fa2, gen_fn_optimized, PROMPTS, "optimized-fa2")
    
    del model_fa2, tok_fa2
    free_model()
except Exception as e:
    print(f"FA2 not available: {e}")
    optimized_fa2_results = None

OPTIMIZED — Flash Attention 2


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  [optimized-fa2] prompt 0: 19 tokens in 46.67s (0.4 tok/s)
  [optimized-fa2] prompt 1: 21 tokens in 4.78s (4.4 tok/s)
  [optimized-fa2] prompt 2: 19 tokens in 9.74s (2.0 tok/s)
  [optimized-fa2] prompt 3: 437 tokens in 61.66s (7.1 tok/s)
  [optimized-fa2] prompt 4: 370 tokens in 38.64s (9.6 tok/s)


## 8. Compare outputs

In [22]:
def compare_outputs(results_a, results_b, label_a, label_b):
    """Compare generated token sequences between two runs."""
    if results_a is None or results_b is None:
        print(f"  Skipping comparison (one run is missing)")
        return
    
    print(f"\n{'─' * 60}")
    print(f"Output comparison: {label_a} vs {label_b}")
    print(f"{'─' * 60}")
    
    all_match = True
    for a, b in zip(results_a, results_b):
        idx = a["prompt_idx"]
        ids_a = a["token_ids"]
        ids_b = b["token_ids"]
        
        if ids_a == ids_b:
            print(f"  Prompt {idx}: EXACT MATCH ({len(ids_a)} tokens)")
        else:
            all_match = False
            # Find first divergence
            min_len = min(len(ids_a), len(ids_b))
            first_diff = min_len  # default: one is longer
            for k in range(min_len):
                if ids_a[k] != ids_b[k]:
                    first_diff = k
                    break
            print(f"  Prompt {idx}: DIVERGE at token {first_diff}  "
                  f"(len {len(ids_a)} vs {len(ids_b)})")
    
    if all_match:
        print(f"\n  ✓ All {len(results_a)} prompts produce identical output")
    else:
        print(f"\n  ⚠ Some outputs differ — review above")


# Same attention backend: baseline vs optimized (should be identical)
compare_outputs(baseline_sdpa_results, optimized_sdpa_results, "baseline-sdpa", "optimized-sdpa")
compare_outputs(baseline_fa2_results, optimized_fa2_results, "baseline-fa2", "optimized-fa2")

# Cross-backend: sdpa vs fa2 (may differ due to numerical precision)
compare_outputs(optimized_sdpa_results, optimized_fa2_results, "optimized-sdpa", "optimized-fa2")


────────────────────────────────────────────────────────────
Output comparison: baseline-sdpa vs optimized-sdpa
────────────────────────────────────────────────────────────
  Prompt 0: EXACT MATCH (19 tokens)
  Prompt 1: EXACT MATCH (21 tokens)
  Prompt 2: EXACT MATCH (19 tokens)
  Prompt 3: EXACT MATCH (406 tokens)
  Prompt 4: EXACT MATCH (29 tokens)

  ✓ All 5 prompts produce identical output

────────────────────────────────────────────────────────────
Output comparison: baseline-fa2 vs optimized-fa2
────────────────────────────────────────────────────────────
  Prompt 0: EXACT MATCH (19 tokens)
  Prompt 1: EXACT MATCH (21 tokens)
  Prompt 2: EXACT MATCH (19 tokens)
  Prompt 3: EXACT MATCH (437 tokens)
  Prompt 4: EXACT MATCH (370 tokens)

  ✓ All 5 prompts produce identical output

────────────────────────────────────────────────────────────
Output comparison: optimized-sdpa vs optimized-fa2
────────────────────────────────────────────────────────────
  Prompt 0: EXACT MATCH (19 t

## 9. Wall time comparison

In [23]:
def summarize_timing(results, label):
    """Return summary stats for a set of benchmark results."""
    if results is None:
        return None
    walls = [r["wall_sec"] for r in results]
    tps = [r["tok_per_sec"] for r in results]
    gens = [r["gen_tokens"] for r in results]
    return {
        "label": label,
        "num_prompts": len(results),
        "total_tokens": sum(gens),
        "total_wall_sec": sum(walls),
        "mean_wall_sec": np.mean(walls),
        "median_wall_sec": np.median(walls),
        "mean_tok_per_sec": np.mean(tps),
        "median_tok_per_sec": np.median(tps),
    }


all_summaries = []
for results, label in [
    (baseline_sdpa_results, "baseline-sdpa"),
    (baseline_fa2_results, "baseline-fa2"),
    (optimized_sdpa_results, "optimized-sdpa"),
    (optimized_fa2_results, "optimized-fa2"),
]:
    s = summarize_timing(results, label)
    if s:
        all_summaries.append(s)

print(f"\n{'Label':<20} {'Prompts':>8} {'Tokens':>8} {'Total(s)':>10} {'Mean(s)':>10} {'Med(s)':>10} {'Mean tok/s':>12} {'Med tok/s':>12}")
print("─" * 100)
for s in all_summaries:
    print(f"{s['label']:<20} {s['num_prompts']:>8} {s['total_tokens']:>8} "
          f"{s['total_wall_sec']:>10.2f} {s['mean_wall_sec']:>10.2f} {s['median_wall_sec']:>10.2f} "
          f"{s['mean_tok_per_sec']:>12.1f} {s['median_tok_per_sec']:>12.1f}")


Label                 Prompts   Tokens   Total(s)    Mean(s)     Med(s)   Mean tok/s    Med tok/s
────────────────────────────────────────────────────────────────────────────────────────────────────
baseline-sdpa               5      494     170.88      34.18      38.09          2.9          2.5
baseline-fa2                5      866     162.39      32.48      39.08          4.6          4.4
optimized-sdpa              5      494     170.02      34.00      37.87          2.9          2.5
optimized-fa2               5      866     161.48      32.30      38.64          4.7          4.4


In [24]:
# Speedup calculations
def calc_speedup(base_results, opt_results, label):
    if base_results is None or opt_results is None:
        return
    base_total = sum(r["wall_sec"] for r in base_results)
    opt_total = sum(r["wall_sec"] for r in opt_results)
    speedup = base_total / opt_total if opt_total > 0 else float("inf")
    print(f"{label}: {base_total:.2f}s → {opt_total:.2f}s  ({speedup:.2f}x speedup)")

print("\n=== Speedup Summary ===")
calc_speedup(baseline_sdpa_results, optimized_sdpa_results, "SDPA")
calc_speedup(baseline_fa2_results, optimized_fa2_results, "FA2")
calc_speedup(baseline_sdpa_results, optimized_fa2_results, "SDPA→FA2 (total)")


=== Speedup Summary ===
SDPA: 170.88s → 170.02s  (1.01x speedup)
FA2: 162.39s → 161.48s  (1.01x speedup)
SDPA→FA2 (total): 170.88s → 161.48s  (1.06x speedup)


## 10. Per-prompt detail

In [25]:
import pandas as pd

all_results = []
for res_list in [baseline_sdpa_results, baseline_fa2_results, optimized_sdpa_results, optimized_fa2_results]:
    if res_list:
        for r in res_list:
            all_results.append({
                "label": r["label"],
                "prompt": PROMPTS[r["prompt_idx"]][:60] + "...",
                "input_len": r["input_len"],
                "gen_tokens": r["gen_tokens"],
                "wall_sec": round(r["wall_sec"], 3),
                "tok_per_sec": round(r["tok_per_sec"], 1),
            })

df = pd.DataFrame(all_results)
df

,label,prompt,input_len,gen_tokens,wall_sec,tok_per_sec
0,baseline-sdpa,Solve: What is the sum of all integer values o...,64,19,39.970,0.5
1,baseline-sdpa,"If $f(x) = 3x^2 - 7$ and $g(x) = x + 1$, what ...",69,21,3.774,5.6
2,baseline-sdpa,Find the value of $x$ if $2^{x+1} = 32$....,53,19,7.689,2.5
3,baseline-sdpa,What is the remainder when $3^{100}$ is divide...,51,406,81.357,5.0
4,baseline-sdpa,Simplify $\sqrt{50} + \sqrt{18}$....,49,29,38.089,0.8
5,baseline-fa2,Solve: What is the sum of all integer values o...,64,19,46.830,0.4
6,baseline-fa2,"If $f(x) = 3x^2 - 7$ and $g(x) = x + 1$, what ...",69,21,4.816,4.4
7,baseline-fa2,Find the value of $x$ if $2^{x+1} = 32$....,53,19,9.766,1.9
8,baseline-fa2,What is the remainder when $3^{100}$ is divide...,51,437,61.901,7.1
9,baseline-fa2,Simplify $\sqrt{50} + \sqrt{18}$....,49,370,39.078,9.5


## 11. Sample outputs (side by side)

In [26]:
# Show first 500 chars of decoded output for each prompt, baseline vs optimized (SDPA)
for i in range(min(3, len(PROMPTS))):
    print(f"\n{'=' * 80}")
    print(f"Prompt {i}: {PROMPTS[i][:80]}")
    print(f"{'=' * 80}")
    
    base = baseline_sdpa_results[i]["decoded"][:500]
    opt = optimized_sdpa_results[i]["decoded"][:500]
    
    print(f"\n--- Baseline SDPA ---")
    print(base)
    print(f"\n--- Optimized SDPA ---")
    print(opt)
    
    if base == opt:
        print("\n  → IDENTICAL")
    else:
        print("\n  → DIFFERS")


Prompt 0: Solve: What is the sum of all integer values of $n$ such that $\frac{20}{2n-1}$ 

--- Baseline SDPA ---
<think>
[THOUGHT] \boxed{2} [RETURN]
</think>
\boxed{2}<|im_end|>

--- Optimized SDPA ---
<think>
[THOUGHT] \boxed{2} [RETURN]
</think>
\boxed{2}<|im_end|>

  → IDENTICAL

Prompt 1: If $f(x) = 3x^2 - 7$ and $g(x) = x + 1$, what is $f(g(2))$?

--- Baseline SDPA ---
<think>
[THOUGHT] \boxed{15} [RETURN]
</think>
\boxed{15}<|im_end|>

--- Optimized SDPA ---
<think>
[THOUGHT] \boxed{15} [RETURN]
</think>
\boxed{15}<|im_end|>

  → IDENTICAL

Prompt 2: Find the value of $x$ if $2^{x+1} = 32$.

--- Baseline SDPA ---
<think>
[THOUGHT] \boxed{4} [RETURN]
</think>
\boxed{4}<|im_end|>

--- Optimized SDPA ---
<think>
[THOUGHT] \boxed{4} [RETURN]
</think>
\boxed{4}<|im_end|>

  → IDENTICAL


## 12. Save results

In [27]:
output = {
    "config": {
        "baseline_rev": BASELINE_REV,
        "optimized_branch": OPTIMIZED_BRANCH,
        "model_id": MODEL_ID,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_samples": NUM_SAMPLES,
        "gpu": torch.cuda.get_device_name(0),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
    },
    "summaries": all_summaries,
}

# Strip token_ids from saved results to keep file small
for label_key, res_list in [
    ("baseline_sdpa", baseline_sdpa_results),
    ("baseline_fa2", baseline_fa2_results),
    ("optimized_sdpa", optimized_sdpa_results),
    ("optimized_fa2", optimized_fa2_results),
]:
    if res_list:
        output[label_key] = [{k: v for k, v in r.items() if k != "token_ids"} for r in res_list]

with open("/content/benchmark_results.json", "w") as f:
    json.dump(output, f, indent=2, default=str)

print("Results saved to /content/benchmark_results.json")

Results saved to /content/benchmark_results.json


In [28]:
output

{'config': {'baseline_rev': '07a3478f3873ec6d8f2abb38c5073abe15033181',
  'optimized_branch': 'claude/optimize-inference-speed-lkCRI',
  'model_id': 'anujjamwal/OpenMath-Nemotron-1.5B-PruneAware',
  'max_new_tokens': 4096,
  'num_samples': 5,
  'gpu': 'NVIDIA A100-SXM4-40GB',
  'torch_version': '2.10.0+cu128',
  'cuda_version': '12.8'},
 'summaries': [{'label': 'baseline-sdpa',
   'num_prompts': 5,
   'total_tokens': 494,
   'total_wall_sec': 170.87804800499998,
   'mean_wall_sec': np.float64(34.175609601),
   'median_wall_sec': np.float64(38.088822874000016),
   'mean_tok_per_sec': np.float64(2.8526056869438206),
   'median_tok_per_sec': np.float64(2.4712135776496336)},
  {'label': 'baseline-fa2',
   'num_prompts': 5,
   'total_tokens': 866,
   'total_wall_sec': 162.39058771799967,
   'mean_wall_sec': np.float64(32.478117543599936),
   'median_wall_sec': np.float64(39.077986295999835),
   'mean_tok_per_sec': np.float64(4.647992546124076),
   'median_tok_per_sec': np.float64(4.36086505